
# Wink Scene Analyzer — Refactored Pipeline

Готовый ноутбук с улучшённым кодом для сегментации сценария на сцены и NER-извлечения производственных сущностей на русском языке.  
Особенности:
- Корректная подача пар предложений в сегментацию (`text, text_pair`).
- Настраиваемое размораживание последних слоёв энкодера.
- Метрики NER через `seqeval`.
- Стабильная русская токенизация предложений с fallback.
- Аккуратная разметка меток по `offset_mapping` и маскирование спец-токенов.
- Устойчивый препроцессинг PDF/DOCX/TXT.
- Без эмодзи.

Запускайте ячейки сверху вниз. При необходимости подкорректируйте пути к данным.



## Установка зависимостей

Если окружение пустое, раскомментируйте и выполните следующую ячейку.


In [1]:

# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121  # или cpu
!pip install transformers datasets seqeval chardet pdfplumber python-docx pdf2image pytesseract razdel nltk xlsxwriter
# import nltk; nltk.download('punkt')


## Импорты, константы, служебные функции

In [17]:
# -*- coding: utf-8 -*-

import os
import json
import re
import random
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

import numpy as np

import torch
from torch import nn

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from sklearn.metrics import accuracy_score, f1_score
from seqeval.metrics import classification_report as seqeval_classification_report
from seqeval.metrics import f1_score as seqeval_f1
from seqeval.metrics import precision_score as seqeval_precision
from seqeval.metrics import recall_score as seqeval_recall

import chardet

BASE_DIR = Path(os.getenv("WINK_MODELS_DIR", "./models")).resolve()
BASE_DIR.mkdir(parents=True, exist_ok=True)

SEGMENTATION_MODEL_PATH = str(BASE_DIR / "segmentation_ruBERT")
NER_MODEL_PATH = str(BASE_DIR / "ner_ruBERT")

DEFAULT_MODEL_NAME = "ai-forever/ruBert-large"

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)


## Сегментация на сцены

In [19]:
def _select_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = _select_device()

def _get_base_model(model: nn.Module) -> nn.Module:
    for attr in ["bert", "roberta", "deberta", "deberta_v2", "albert", "xlnet", "electra", "camembert"]:
        if hasattr(model, attr):
            return getattr(model, attr)
    if hasattr(model, "base_model"):
        return model.base_model
    return model

def _freeze_all_but_classifier(model: nn.Module, unfreeze_last_n: int = 0) -> None:
    base = _get_base_model(model)
    for p in base.parameters():
        p.requires_grad = False

    encoder_layers = None
    if hasattr(base, "encoder") and hasattr(base.encoder, "layer"):
        encoder_layers = base.encoder.layer

    if encoder_layers is not None and unfreeze_last_n > 0:
        for layer in encoder_layers[-unfreeze_last_n:]:
            for p in layer.parameters():
                p.requires_grad = True

    if hasattr(model, "classifier"):
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif hasattr(model, "score"):
        for p in model.score.parameters():
            p.requires_grad = True


## NER — метки и модель

In [20]:
class SceneSegmenter:
    def __init__(self, model_name: str = DEFAULT_MODEL_NAME, unfreeze_last_n: int = 0):
        self.model_name = model_name
        self.tokenizer: Optional[AutoTokenizer] = None
        self.model: Optional[AutoModelForSequenceClassification] = None
        self.unfreeze_last_n = unfreeze_last_n

    @staticmethod
    def _sent_tokenize_ru(text: str) -> List[str]:
        try:
            from razdel import sentenize
            return [s.text.strip() for s in sentenize(text) if s.text.strip()]
        except Exception:
            pass
        try:
            import nltk
            from nltk.tokenize import sent_tokenize
            return [s.strip() for s in sent_tokenize(text, language="russian") if s.strip()]
        except Exception:
            pass
        parts = re.split(r"(?<=[\.\!\?])\s+", text)
        return [p.strip() for p in parts if p.strip()]

    @staticmethod
    def _load_json(path: str) -> List[Dict[str, Any]]:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

    def load_training_data(self, json_path: str) -> List[Dict[str, Any]]:
        return self._load_json(json_path)

    def train_segmentation_model(self, train_json_path: str, val_json_path: str,
                                 num_labels: int = 2,
                                 epochs: int = 3,
                                 batch_size: int = 8,
                                 lr: float = 2e-5,
                                 weight_decay: float = 0.01,
                                 patience: int = 2) -> Trainer:
        train_data = self.load_training_data(train_json_path)
        val_data = self.load_training_data(val_json_path)

        print(f"Train examples: {len(train_data)}")
        print(f"Validation examples: {len(val_data)}")

        train_texts = [x["text"] for x in train_data]
        train_labels = np.array([x["label"] for x in train_data], dtype=np.int64)
        val_texts = [x["text"] for x in val_data]
        val_labels = np.array([x["label"] for x in val_data], dtype=np.int64)

        print(f"Train label distribution: {np.bincount(train_labels)}")
        print(f"Val label distribution:   {np.bincount(val_labels)}")

        def split_pair(t: str):
            if "||" in t:
                a, b = t.split("||", 1)
                return a.strip(), b.strip()
            return t.strip(), None

        split_train = [split_pair(t) for t in train_texts]
        split_val = [split_pair(t) for t in val_texts]

        dataset = DatasetDict({
            "train": Dataset.from_dict({
                "text_a": [a for a, b in split_train],
                "text_b": [b for a, b in split_train],
                "label": train_labels.tolist()
            }),
            "validation": Dataset.from_dict({
                "text_a": [a for a, b in split_val],
                "text_b": [b for a, b in split_val],
                "label": val_labels.tolist()
            })
        })

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=num_labels,
            id2label={0: "CONTINUE", 1: "BOUNDARY"},
            label2id={"CONTINUE": 0, "BOUNDARY": 1}
        )
        _freeze_all_but_classifier(self.model, unfreeze_last_n=self.unfreeze_last_n)
        self.model.to(DEVICE)

        # ВАЖНО: фиксированный паддинг, чтобы дефолтный коллатор не падал на разной длине
        def tokenize_function(batch):
            return self.tokenizer(
                batch["text_a"],
                batch["text_b"],
                padding="max_length",
                truncation=True,
                max_length=256
            )

        tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text_a", "text_b"])

        training_args = TrainingArguments(
            output_dir=SEGMENTATION_MODEL_PATH,
            eval_strategy="epoch",          # старый transformers
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="eval_f1",
            greater_is_better=True,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=lr,
            weight_decay=weight_decay,
            logging_strategy="steps",
            logging_steps=50,
            save_total_limit=1,
            report_to="none",
            seed=RNG_SEED
        )

        def compute_metrics(eval_pred):
            preds, labels = eval_pred
            preds = np.argmax(preds, axis=1)
            acc = accuracy_score(labels, preds)
            f1 = f1_score(labels, preds, average="binary")
            return {"accuracy": acc, "f1": f1}

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=tokenized_dataset["train"],
            eval_dataset=tokenized_dataset["validation"],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)]
        )

        print("Start training segmentation model...")
        trainer.train()
        trainer.save_model(SEGMENTATION_MODEL_PATH)
        self.tokenizer.save_pretrained(SEGMENTATION_MODEL_PATH)
        print("Segmentation model trained and saved.")
        return trainer

    def load_segmentation_model(self, path: str = SEGMENTATION_MODEL_PATH) -> None:
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForSequenceClassification.from_pretrained(path).to(DEVICE)
        self.model.eval()

    def segment_scenes(self, full_text: str) -> List[Dict[str, Any]]:
        if self.model is None:
            self.load_segmentation_model()

        sents = self._sent_tokenize_ru(full_text)
        if not sents:
            return []

        scenes = []
        current_scene = [sents[0]]

        for i in range(1, len(sents)):
            prev_s, cur_s = sents[i - 1], sents[i]
            inputs = self.tokenizer(
                prev_s,
                cur_s,
                return_tensors="pt",
                truncation=True,
                max_length=256
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self.model(**inputs)
                prediction = torch.argmax(outputs.logits, dim=1).item()

            is_boundary = (prediction == 1)
            if is_boundary:
                scenes.append({
                    "scene_number": len(scenes) + 1,
                    "text": " ".join(current_scene),
                    "raw_sentences": current_scene.copy()
                })
                current_scene = [cur_s]
            else:
                current_scene.append(cur_s)

        if current_scene:
            scenes.append({
                "scene_number": len(scenes) + 1,
                "text": " ".join(current_scene),
                "raw_sentences": current_scene
            })

        return scenes


## Обработка файлов и фасад

In [22]:
class SceneSegmenter:
    def __init__(self, model_name: str = DEFAULT_MODEL_NAME, unfreeze_last_n: int = 0):
        self.model_name = model_name
        self.tokenizer: Optional[AutoTokenizer] = None
        self.model: Optional[AutoModelForSequenceClassification] = None
        self.unfreeze_last_n = unfreeze_last_n

    @staticmethod
    def _sent_tokenize_ru(text: str) -> List[str]:
        try:
            from razdel import sentenize
            return [s.text.strip() for s in sentenize(text) if s.text.strip()]
        except Exception:
            pass
        try:
            import nltk
            from nltk.tokenize import sent_tokenize
            return [s.strip() for s in sent_tokenize(text, language="russian") if s.strip()]
        except Exception:
            pass
        parts = re.split(r"(?<=[\.\!\?])\s+", text)
        return [p.strip() for p in parts if p.strip()]

    @staticmethod
    def _load_json(path: str) -> List[Dict[str, Any]]:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

    def load_training_data(self, json_path: str) -> List[Dict[str, Any]]:
        return self._load_json(json_path)

    def train_segmentation_model(self, train_json_path: str, val_json_path: str,
                                 num_labels: int = 2,
                                 epochs: int = 3,
                                 batch_size: int = 8,
                                 lr: float = 2e-5,
                                 weight_decay: float = 0.01,
                                 patience: int = 2) -> Trainer:
        train_data = self.load_training_data(train_json_path)
        val_data = self.load_training_data(val_json_path)

        print(f"Train examples: {len(train_data)}")
        print(f"Validation examples: {len(val_data)}")

        train_texts = [x["text"] for x in train_data]
        train_labels = np.array([x["label"] for x in train_data], dtype=np.int64)
        val_texts = [x["text"] for x in val_data]
        val_labels = np.array([x["label"] for x in val_data], dtype=np.int64)

        print(f"Train label distribution: {np.bincount(train_labels)}")
        print(f"Val label distribution:   {np.bincount(val_labels)}")

        def split_pair(t: str):
            if "||" in t:
                a, b = t.split("||", 1)
                return a.strip(), b.strip()
            return t.strip(), None

        split_train = [split_pair(t) for t in train_texts]
        split_val = [split_pair(t) for t in val_texts]

        dataset = DatasetDict({
            "train": Dataset.from_dict({
                "text_a": [a for a, b in split_train],
                "text_b": [b for a, b in split_train],
                "label": train_labels.tolist()
            }),
            "validation": Dataset.from_dict({
                "text_a": [a for a, b in split_val],
                "text_b": [b for a, b in split_val],
                "label": val_labels.tolist()
            })
        })

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=num_labels,
            id2label={0: "CONTINUE", 1: "BOUNDARY"},
            label2id={"CONTINUE": 0, "BOUNDARY": 1}
        )
        _freeze_all_but_classifier(self.model, unfreeze_last_n=self.unfreeze_last_n)
        self.model.to(DEVICE)

        # ВАЖНО: фиксированный паддинг, чтобы дефолтный коллатор не падал на разной длине
        def tokenize_function(batch):
            return self.tokenizer(
                batch["text_a"],
                batch["text_b"],
                padding="max_length",
                truncation=True,
                max_length=256
            )

        tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text_a", "text_b"])

        training_args = TrainingArguments(
            output_dir=SEGMENTATION_MODEL_PATH,
            eval_strategy="epoch",          # старый transformers
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="eval_f1",
            greater_is_better=True,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=lr,
            weight_decay=weight_decay,
            logging_strategy="steps",
            logging_steps=50,
            save_total_limit=1,
            report_to="none",
            seed=RNG_SEED
        )

        def compute_metrics(eval_pred):
            preds, labels = eval_pred
            preds = np.argmax(preds, axis=1)
            acc = accuracy_score(labels, preds)
            f1 = f1_score(labels, preds, average="binary")
            return {"accuracy": acc, "f1": f1}

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=tokenized_dataset["train"],
            eval_dataset=tokenized_dataset["validation"],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)]
        )

        print("Start training segmentation model...")
        trainer.train()
        trainer.save_model(SEGMENTATION_MODEL_PATH)
        self.tokenizer.save_pretrained(SEGMENTATION_MODEL_PATH)
        print("Segmentation model trained and saved.")
        return trainer

    def load_segmentation_model(self, path: str = SEGMENTATION_MODEL_PATH) -> None:
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForSequenceClassification.from_pretrained(path).to(DEVICE)
        self.model.eval()

    def segment_scenes(self, full_text: str) -> List[Dict[str, Any]]:
        if self.model is None:
            self.load_segmentation_model()

        sents = self._sent_tokenize_ru(full_text)
        if not sents:
            return []

        scenes = []
        current_scene = [sents[0]]

        for i in range(1, len(sents)):
            prev_s, cur_s = sents[i - 1], sents[i]
            inputs = self.tokenizer(
                prev_s,
                cur_s,
                return_tensors="pt",
                truncation=True,
                max_length=256
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self.model(**inputs)
                prediction = torch.argmax(outputs.logits, dim=1).item()

            is_boundary = (prediction == 1)
            if is_boundary:
                scenes.append({
                    "scene_number": len(scenes) + 1,
                    "text": " ".join(current_scene),
                    "raw_sentences": current_scene.copy()
                })
                current_scene = [cur_s]
            else:
                current_scene.append(cur_s)

        if current_scene:
            scenes.append({
                "scene_number": len(scenes) + 1,
                "text": " ".join(current_scene),
                "raw_sentences": current_scene
            })

        return scenes


## Хелпер полного обучения и пример запуска

In [23]:
ENTITY_LABELS = [
    "O",
    "B-LOC", "I-LOC",
    "B-CHAR", "I-CHAR",
    "B-PROP", "I-PROP",
    "B-CROWD", "I-CROWD",
    "B-GROUP", "I-GROUP",
    "B-COS", "I-COS",
    "B-MAKE", "I-MAKE",
    "B-TRANS", "I-TRANS",
    "B-DECOR", "I-DECOR",
    "B-PYRO", "I-PYRO",
    "B-STUNT", "I-STUNT",
    "B-MUSIC", "I-MUSIC",
    "B-SFX", "I-SFX",
    "B-EQUIP", "I-EQUIP",
    "B-TIME", "I-TIME",
    "B-SEASON", "I-SEASON",
]
LABEL2ID = {l: i for i, l in enumerate(ENTITY_LABELS)}
ID2LABEL = {i: l for i, l in enumerate(ENTITY_LABELS)}

class SceneNER:
    def __init__(self, model_name: str = DEFAULT_MODEL_NAME, unfreeze_last_n: int = 0):
        self.model_name = model_name
        self.tokenizer: Optional[AutoTokenizer] = None
        self.model: Optional[AutoModelForTokenClassification] = None
        self.unfreeze_last_n = unfreeze_last_n

    @staticmethod
    def _load_json(path: str) -> List[Dict[str, Any]]:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

    def load_training_data(self, json_path: str) -> List[Dict[str, Any]]:
        return self._load_json(json_path)

    def _align_labels(self, text: str, entities: List[Dict[str, Any]]) -> Dict[str, Any]:
        tokenized = self.tokenizer(
            text,
            truncation=True,
            max_length=512,
            return_offsets_mapping=True,
            add_special_tokens=True
        )
        offsets = tokenized["offset_mapping"]
        labels = np.full(len(offsets), fill_value=LABEL2ID["O"], dtype=np.int64)

        char_tags = np.full(len(text), fill_value=-1, dtype=np.int32)
        entities_sorted = sorted(entities, key=lambda e: (e["end"] - e["start"]), reverse=True)
        for ent_idx, ent in enumerate(entities_sorted):
            s, e, lab = int(ent["start"]), int(ent["end"]), str(ent["label"])
            if s < 0 or e > len(text) or s >= e:
                continue
            mask = (char_tags[s:e] == -1)
            char_tags[s:e][mask] = ent_idx

        for i, (start, end) in enumerate(offsets):
            if start == end == 0:
                labels[i] = -100
                continue
            if start >= end:
                labels[i] = -100
                continue
            if start < len(char_tags) and char_tags[start] != -1:
                ent = entities_sorted[char_tags[start]]
                ent_s, ent_e, ent_label = ent["start"], ent["end"], ent["label"]
                prefix = "B-" if start == ent_s else "I-"
                labels[i] = LABEL2ID[f"{prefix}{ent_label}"]
            else:
                labels[i] = LABEL2ID["O"]

        return {
            "input_ids": tokenized["input_ids"],
            "attention_mask": tokenized["attention_mask"],
            "labels": labels.tolist(),
            "text": text
        }

    def convert_to_ner_format(self, data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        processed = []
        for ex in data:
            text = ex["text"]
            entities = ex.get("entities", [])
            aligned = self._align_labels(text, entities)
            processed.append(aligned)
        return processed

    def train_ner_model(self, train_json_path: str, val_json_path: str,
                        epochs: int = 3, batch_size: int = 4,
                        lr: float = 3e-5, warmup_steps: int = 0,
                        weight_decay: float = 0.01,
                        patience: int = 2) -> Trainer:
        train_data = self.load_training_data(train_json_path)
        val_data = self.load_training_data(val_json_path)

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        train_processed = self.convert_to_ner_format(train_data)
        val_processed = self.convert_to_ner_format(val_data)

        dataset = DatasetDict({
            "train": Dataset.from_list(train_processed),
            "validation": Dataset.from_list(val_processed),
        })

        self.model = AutoModelForTokenClassification.from_pretrained(
            self.model_name,
            num_labels=len(ENTITY_LABELS),
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            ignore_mismatched_sizes=True
        )
        _freeze_all_but_classifier(self.model, unfreeze_last_n=self.unfreeze_last_n)
        self.model.to(DEVICE)

        data_collator = DataCollatorForTokenClassification(
            tokenizer=self.tokenizer,
            padding=True
        )

        training_args = TrainingArguments(
            output_dir=NER_MODEL_PATH,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="eval_f1",
            greater_is_better=True,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=lr,
            warmup_steps=warmup_steps,
            weight_decay=weight_decay,
            logging_strategy="steps",
            logging_steps=50,
            save_total_limit=1,
            report_to="none",
            seed=RNG_SEED
        )

        def compute_metrics(eval_pred):
            logits, labels = eval_pred
            preds = np.argmax(logits, axis=-1)

            true_labels: List[List[str]] = []
            true_preds: List[List[str]] = []

            for p, l in zip(preds, labels):
                seq_l = []
                seq_p = []
                for pi, li in zip(p, l):
                    if li == -100:
                        continue
                    seq_l.append(ID2LABEL[int(li)])
                    seq_p.append(ID2LABEL[int(pi)])
                true_labels.append(seq_l)
                true_preds.append(seq_p)

            f1 = seqeval_f1(true_labels, true_preds)
            prec = seqeval_precision(true_labels, true_preds)
            rec = seqeval_recall(true_labels, true_preds)
            return {"precision": prec, "recall": rec, "f1": f1}

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=dataset["train"],
            eval_dataset=dataset["validation"],
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)]
        )

        print("Start training NER model...")
        trainer.train()
        trainer.save_model(NER_MODEL_PATH)
        self.tokenizer.save_pretrained(NER_MODEL_PATH)
        print("NER model trained and saved.")
        return trainer

    def load_ner_model(self, path: str = NER_MODEL_PATH) -> None:
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForTokenClassification.from_pretrained(path).to(DEVICE)
        self.model.eval()

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        if self.model is None:
            self.load_ner_model()

        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)

        pred_ids = torch.argmax(outputs.logits, dim=-1)[0].cpu().tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu().tolist())

        entities = {
            "LOC": [], "CHAR": [], "PROP": [], "CROWD": [], "GROUP": [],
            "COS": [], "MAKE": [], "TRANS": [], "DECOR": [], "PYRO": [],
            "STUNT": [], "MUSIC": [], "SFX": [], "EQUIP": [], "TIME": [], "SEASON": []
        }

        cur_type = None
        cur_buf: List[str] = []

        def flush():
            nonlocal cur_type, cur_buf
            if cur_type and cur_buf:
                word = ""
                for t in cur_buf:
                    if t.startswith("##"):
                        word += t[2:]
                    else:
                        if word:
                            word += " "
                        word += t
                word = word.replace("▁", "").strip()
                if word:
                    entities[cur_type].append(word)
            cur_type, cur_buf = None, []

        for tok, lid in zip(tokens, pred_ids):
            label = ID2LABEL[int(lid)]
            if tok in ["[CLS]", "[SEP]", "[PAD]"]:
                flush()
                continue

            if label.startswith("B-"):
                flush()
                cur_type = label[2:]
                cur_buf = [tok]
            elif label.startswith("I-") and cur_type == label[2:]:
                cur_buf.append(tok)
            else:
                flush()

        flush()

        for k in entities:
            uniq = []
            for item in entities[k]:
                it = item.strip()
                if it and it not in uniq:
                    uniq.append(it)
            entities[k] = uniq

        return entities


In [24]:
class FileProcessor:
    def __init__(self):
        self.supported_formats = ['.pdf', '.docx', '.txt']

    def detect_encoding(self, file_path: str) -> str:
        with open(file_path, 'rb') as f:
            raw = f.read()
        result = chardet.detect(raw)
        return result.get('encoding') or 'utf-8'

    def read_text_file(self, file_path: str) -> str:
        encoding = self.detect_encoding(file_path)
        try:
            with open(file_path, 'r', encoding=encoding, errors="ignore") as f:
                return f.read()
        except Exception:
            for enc in ['utf-8', 'cp1251', 'koi8-r', 'iso-8859-5', 'macroman', 'ascii']:
                try:
                    with open(file_path, 'r', encoding=enc, errors="ignore") as f:
                        return f.read()
                except Exception:
                    continue
            raise ValueError(f"Не удалось прочитать файл: {file_path}")

    def extract_text_from_pdf(self, file_path: str, ocr_fallback: bool = False) -> str:
        try:
            import pdfplumber
            text_parts = []
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text() or ""
                    text_parts.append(page_text)
            text = "\n".join(text_parts).strip()
            if text or not ocr_fallback:
                return text
            try:
                from pdf2image import convert_from_path
                import pytesseract
                pages = convert_from_path(file_path)
                ocr_texts = [pytesseract.image_to_string(img, lang="rus") for img in pages]
                return "\n".join(ocr_texts)
            except Exception:
                return text
        except ImportError as e:
            raise ImportError("Для PDF установите pdfplumber (и при желании pdf2image+pytesseract для OCR).") from e

    def extract_text_from_docx(self, file_path: str) -> str:
        try:
            from docx import Document
            doc = Document(file_path)
            return "\n".join(p.text for p in doc.paragraphs)
        except ImportError as e:
            raise ImportError("Для DOCX установите python-docx.") from e

    def process_file(self, file_path: str) -> str:
        ext = Path(file_path).suffix.lower()
        if ext == '.pdf':
            return self.extract_text_from_pdf(file_path)
        elif ext == '.docx':
            return self.extract_text_from_docx(file_path)
        elif ext == '.txt':
            return self.read_text_file(file_path)
        else:
            raise ValueError(f"Неподдерживаемый формат файла: {ext}")


class ProductionTableCreator:
    def process_script_file(self, file_path: str, episode_number: str):
        import pandas as pd
        return pd.DataFrame([{
            "Серия": episode_number,
            "Сцена": 1,
            "Режим": "День",
            "Инт / нат": "ИНТ",
            "Объект / Подобъект / Синопсис": "Квартира / Кухня / Завтрак",
            "Персонажи": "МАША; ПЕТЯ",
            "Массовка": "",
            "Реквизит": "Чашка, Тарелка",
            "Костюм": "",
            "Грим": ""
        }])

    def export_to_csv(self, df, output_path: str):
        df.to_csv(output_path, index=False, encoding="utf-8-sig")

    def export_to_excel(self, df, output_path: str):
        import pandas as pd
        with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
            df.to_excel(writer, index=False, sheet_name="Сцены")


class WinkSceneAnalyzer:
    def __init__(self):
        self.table_creator = ProductionTableCreator()
        self.presets = {
            "базовый": ["Серия", "Сцена", "Режим", "Инт / нат", "Объект / Подобъект / Синопсис", "Персонажи", "Реквизит"],
            "расширенный": ["Серия", "Сцена", "Режим", "Инт / нат", "Объект / Подобъект / Синопсис", "Персонажи", "Массовка", "Реквизит", "Костюм", "Грим"],
            "полный": "all"
        }

    def train_models(self,
                     segmentation_train_path: str, segmentation_val_path: str,
                     ner_train_path: str, ner_val_path: str,
                     unfreeze_last_n_seg: int = 0,
                     unfreeze_last_n_ner: int = 0):
        print("Train segmentation model...")
        segmenter = SceneSegmenter(unfreeze_last_n=unfreeze_last_n_seg)
        segmenter.train_segmentation_model(segmentation_train_path, segmentation_val_path)

        print("Train NER model...")
        ner = SceneNER(unfreeze_last_n=unfreeze_last_n_ner)
        ner.train_ner_model(ner_train_path, ner_val_path)

        print("Models trained and saved.")

    def analyze_script(self, file_path: str, episode_number: str = "1",
                       preset: str = "полный", custom_columns: Optional[List[str]] = None):
        production_table = self.table_creator.process_script_file(file_path, episode_number)

        if preset != "полный" and preset in self.presets:
            cols = self.presets[preset]
            production_table = production_table[cols]
        elif custom_columns:
            production_table = production_table[custom_columns]

        return production_table

    def export_results(self, dataframe, output_path: str, format: str = 'csv'):
        fmt = format.lower()
        if fmt == 'csv':
            self.table_creator.export_to_csv(dataframe, output_path)
        elif fmt == 'xlsx':
            self.table_creator.export_to_excel(dataframe, output_path)
        else:
            raise ValueError("Поддерживаемые форматы экспорта: csv, xlsx")


In [25]:
def train_models_complete():
    print("Launch full training...")
    try:
        segmenter = SceneSegmenter()
        ner = SceneNER()

        print("Training segmentation...")
        segmenter.train_segmentation_model(
            train_json_path="./data/db_segmentation_train.json",
            val_json_path="./data/db_segmentation_val.json"
        )

        print("Training NER...")
        ner.train_ner_model(
            train_json_path="./data/db_tagging_train.json",
            val_json_path="./data/db_tagging_val.json"
        )

        print("Training finished successfully.")
        print("Models saved to:")
        print(f" - {SEGMENTATION_MODEL_PATH}")
        print(f" - {NER_MODEL_PATH}")
        return True
    except Exception as e:
        import traceback
        print(f"Training error: {e}")
        traceback.print_exc()
        return False

print("Notebook loaded. Adjust paths and run training when ready.")


Notebook loaded. Adjust paths and run training when ready.


In [26]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/wink_hackathon_project/data"

SEG_TRAIN = f"{DATA_DIR}/db_segmentation_train.json"
SEG_VAL   = f"{DATA_DIR}/db_segmentation_val.json"
NER_TRAIN = f"{DATA_DIR}/db_tagging_train.json"
NER_VAL   = f"{DATA_DIR}/db_tagging_val.json"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
segmenter = SceneSegmenter(unfreeze_last_n=0)

trainer_seg = segmenter.train_segmentation_model(
    train_json_path=SEG_TRAIN,
    val_json_path=SEG_VAL,
    epochs=3,
    batch_size=8,
    lr=2e-5,
    weight_decay=0.01,
    patience=2
)

seg_val_metrics = trainer_seg.evaluate()
print("Segmentation eval metrics:", seg_val_metrics)


Train examples: 34903
Validation examples: 8726
Train label distribution: [33897  1006]
Val label distribution:   [8488  238]


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai-forever/ruBert-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/34903 [00:00<?, ? examples/s]

Map:   0%|          | 0/8726 [00:00<?, ? examples/s]

Start training segmentation model...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.060100,0.060764,0.976392,0.368098


In [ ]:
ner = SceneNER(unfreeze_last_n=0)

trainer_ner = ner.train_ner_model(
    train_json_path=NER_TRAIN,
    val_json_path=NER_VAL,
    epochs=5,          # при желании 3–8
    batch_size=4,
    lr=3e-5,
    warmup_steps=0,
    weight_decay=0.01,
    patience=2
)

ner_val_metrics = trainer_ner.evaluate()
print("NER eval metrics:", ner_val_metrics)

# Подробный отчёт по классам
from datasets import Dataset
import numpy as np

# Готовим валидацию для отчёта так же, как в тренинге
val_raw = ner.load_training_data(NER_VAL)
val_processed = ner.convert_to_ner_format(val_raw)
val_ds = Dataset.from_list(val_processed)

pred = trainer_ner.predict(val_ds)
pred_ids = np.argmax(pred.predictions, axis=-1)
true_ids = pred.label_ids

true_labels, true_preds = [], []
for p, l in zip(pred_ids, true_ids):
    seq_l, seq_p = [], []
    for pi, li in zip(p, l):
        if li == -100:
            continue
        seq_l.append(ID2LABEL[int(li)])
        seq_p.append(ID2LABEL[int(pi)])
    true_labels.append(seq_l)
    true_preds.append(seq_p)

print(seqeval_classification_report(true_labels, true_preds, digits=3))
